# 03 — Manuscript claims

**Every quantitative claim the USRSE'26 manuscript makes about the *LLM
experiment*, recomputed from `analysis/aggregated.csv` with its provenance.**

This notebook is the single source of truth for the numbers in the paper's
*Evaluation* (specification-accuracy & verbosity) sections and the related
*Discussion* points. Each `claim(...)` prints the paper-facing statement, the
computed value, and exactly which columns + filters produced it — so a reviewer
can trace any number in the manuscript back to the data in one place.

It uses the **same filters and pass/accurate definitions as
`01_headline.ipynb`** (the figure notebook), so the claims and the figures
cannot drift.

**Out of scope (not derivable here):** the *wall-clock benchmark* numbers
(23% / 45% faster, the `m7i.2xlarge` runtime experiment) come from a separate
hand-written Josh-vs-Mesa benchmark, **not** from this LLM panel — see the
final cell. The panel's own runtime (LLM-authored models) *is* computed here.


In [ ]:
import os, re, csv, statistics as st
from collections import Counter

# cwd-robust: notebook runs from analysis/ (pixi run lab) or repo root.
AGG = next(p for p in ["aggregated.csv", "analysis/aggregated.csv"] if os.path.exists(p))

# --- canonical scope, identical to 01_headline.ipynb ---
MODEL_ORDER  = ["sonnet","gemma","kimi","minimax","mistral","glm","qwen","nemotron","deepseek"]
TARGET_ORDER = ["mesa","josh","josh-mcp"]
BATCH_FILTER = ["headline-stage1-20260601","headline-rerep-202606020525",
                "headline-rerep-202606021512","fill-*"]
_PAT = re.compile("^(" + "|".join(b.replace("*",".*") for b in BATCH_FILTER) + ")$")

def _bool(x): return str(x).strip().lower() == "true"
def _num(x):
    try: return float(x)
    except (TypeError, ValueError): return None

rows = []
for r in csv.DictReader(open(AGG)):
    if not _PAT.match(r["batch_tag"]): continue
    if r["model"] not in MODEL_ORDER or r["target"] not in TARGET_ORDER: continue
    rows.append(r)

# attempts = drop the synthetic no_scorer rows (provider-throttle / never-started;
# deadline kills were rescored to did_run=False and ARE kept). Matches Panel A.
att = [r for r in rows if r["engagement_status"] != "no_scorer"]
for r in att:
    r["PASS"] = _bool(r["substantive_conformance"]) and _bool(r["did_run"])
    r["ACC"]  = r["PASS"] and _bool(r["regression_fit_ok"])   # "accurate" = pass + regression in-band

def median(xs): return st.median(xs) if xs else float("nan")

# --- claim ledger ---
CLAIMS = []
def claim(tag, statement, value, source):
    CLAIMS.append({"tag": tag, "statement": statement, "value": value, "source": source})
    print(f"[{tag}]  {statement}")
    print(f"    => {value}")
    print(f"    source: {source}\n")

print(f"loaded {len(rows)} rows ({len(att)} attempts) from {AGG}")


## §Evaluation — panel composition

In [ ]:
combo = Counter((r["model"], r["target"]) for r in rows)
rep_counts = Counter(combo.values())

claim("panel.models", "Nine agentic LLMs were evaluated.",
      f"{len(MODEL_ORDER)} models: {', '.join(MODEL_ORDER)}",
      "distinct `model` in MODEL_ORDER (olmo excluded: failed across the board)")

claim("panel.targets", "Three modelling targets per model.",
      f"{len(TARGET_ORDER)}: {', '.join(TARGET_ORDER)}",
      "distinct `target` in TARGET_ORDER")

claim("panel.reps",
      "Replicates per (model x target) combo (manuscript text/caption must say this, NOT 'five').",
      f"{dict(rep_counts)} -> {len(combo)} combos, predominantly 8 reps each (2 combos reached 10 via extra fill waves)",
      "count of rows per (model,target) after BATCH_FILTER")

claim("panel.total_cells",
      "Total agent cells in the analysed panel.",
      f"{len(rows)} cells ({len(att)} real attempts after dropping {len(rows)-len(att)} no_scorer infra rows)",
      "rows after BATCH_FILTER; attempts drop engagement_status=='no_scorer'")


## §Evaluation — correctness & capability tiers

In [ ]:
# pass / accurate by target
for t in TARGET_ORDER:
    sub = [r for r in att if r["target"] == t]
    p = sum(r["PASS"] for r in sub); a = sum(r["ACC"] for r in sub)
    claim(f"pass.{t}",
          f"Pass rate for target '{t}' (used framework AND ran cleanly).",
          f"{p}/{len(sub)} = {100*p/len(sub):.0f}% pass | accurate (+regression) {a}/{len(sub)} = {100*a/len(sub):.0f}%",
          "PASS = substantive_conformance & did_run; ACC adds regression_fit_ok; excl no_scorer")

# per-model overall pass rate -> tiers
rates = []
for m in MODEL_ORDER:
    sub = [r for r in att if r["model"] == m]
    p = sum(r["PASS"] for r in sub)
    rates.append((m, p, len(sub), p/len(sub)))
rates.sort(key=lambda x: -x[3])
reliable = [m for m,_,_,rt in rates if rt >= 0.80]
variable = [m for m,_,_,rt in rates if 0.30 <= rt < 0.80]
failing  = [m for m,_,_,rt in rates if rt < 0.30]
ladder = " | ".join(f"{m} {100*rt:.0f}%" for m,_,_,rt in rates)

claim("tiers.ladder", "Per-model overall pass rate (drives the capability tiers).",
      ladder, "mean(PASS) per model over all targets")
claim("tiers.reliable", "Reliable tier (>=80% pass): completes nearly every cell.",
      ", ".join(reliable), "pass rate >= 0.80")
claim("tiers.variable", "Variable tier (30-80%): succeeds inconsistently.",
      ", ".join(variable), "0.30 <= pass rate < 0.80")
claim("tiers.failing", "Failing tier (<30%): rarely completes given any target.",
      ", ".join(failing), "pass rate < 0.30")
claim("tiers.openweight",
      "Open-weight models lead the top tier (deepseek/glm out-rank sonnet; no statistical power).",
      f"top: {rates[0][0]} {100*rates[0][3]:.0f}%, {rates[1][0]} {100*rates[1][3]:.0f}%, sonnet {100*dict((m,rt) for m,_,_,rt in rates)['sonnet']:.0f}%",
      "per-model pass-rate ranking")


In [ ]:
# josh-mcp penalty vs josh (the reframed 'too constrained' story)
def rate(target):
    sub = [r for r in att if r["target"] == target]
    return sum(r["PASS"] for r in sub), len(sub)
pj, nj = rate("josh"); pm, nm = rate("josh-mcp")
gap = 100*pj/nj - 100*pm/nm
claim("mcp.penalty",
      "Constraining Josh to MCP (no bash) carries a real pass-rate penalty vs bash-enabled Josh.",
      f"josh {100*pj/nj:.0f}% -> josh-mcp {100*pm/nm:.0f}%  (down {gap:.0f} points)",
      "PASS rate josh vs josh-mcp; excl no_scorer")

# per (model,target) pass table (supports 'sonnet/minimax josh>mesa' aside)
print("per (model,target) pass k/n:")
for m in MODEL_ORDER:
    cells = []
    for t in TARGET_ORDER:
        sub = [r for r in att if r["model"]==m and r["target"]==t]
        cells.append(f"{t}={sum(r['PASS'] for r in sub)}/{len(sub)}")
    print(f"  {m:9} " + "  ".join(cells))


## §Evaluation — conciseness (LOC & entropy)

In [ ]:
# Headline conciseness claim: Josh vs Mesa LOC & entropy over ACCURATE reps
# (manuscript wording: 'specification-accurate ... models'). Also report the
# 'passing' definition so the sensitivity to the gate is explicit.
def loc_ent(target, gate):
    locs = [_num(r["src_loc"])     for r in att if r["target"]==target and r[gate] and _num(r["src_loc"]) is not None]
    ents = [_num(r["entropy_bits"]) for r in att if r["target"]==target and r[gate] and _num(r["entropy_bits"]) is not None]
    return locs, ents

for gate, label in [("ACC", "specification-accurate"), ("PASS", "passing")]:
    ml, me = loc_ent("mesa", gate); jl, je = loc_ent("josh", gate)
    mloc, ment, jloc, jent = median(ml), median(me), median(jl), median(je)
    claim(f"loc.josh_vs_mesa.{gate}",
          f"Josh models have far fewer LOC than Mesa ({label} reps, median).",
          f"josh {jloc:.0f} vs mesa {mloc:.0f} LOC -> {100*(1-jloc/mloc):.0f}% fewer | "
          f"entropy josh {jent:.0f} vs mesa {ment:.0f} -> {100*(1-jent/ment):.0f}% less",
          f"median(src_loc) & median(entropy_bits) over {label} reps (gate={gate}); "
          f"LOC excludes comments/imports per harness/loc.py")

# variability: Mesa more variable in LOC than Josh
def iqr(target):
    xs = sorted(_num(r["src_loc"]) for r in att if r["target"]==target and r["PASS"] and _num(r["src_loc"]) is not None)
    if len(xs) < 4: return None
    q1, q3 = xs[len(xs)//4], xs[3*len(xs)//4]
    return q1, q3, q3-q1, st.pstdev(xs)/st.mean(xs)
mi, ji = iqr("mesa"), iqr("josh")
claim("loc.variability",
      "Mesa LOC is more variable than Josh (wider IQR) — more ways to write a valid general-framework model.",
      f"mesa IQR=[{mi[0]:.0f},{mi[1]:.0f}] width={mi[2]:.0f} (CV {mi[3]:.2f}) vs "
      f"josh IQR=[{ji[0]:.0f},{ji[1]:.0f}] width={ji[2]:.0f} (CV {ji[3]:.2f})",
      "IQR & CV of src_loc over passing reps")


## §Discussion — documentation dependence (the llms-full.txt result)

In [ ]:
# doc-fetch: cells that made >=1 doc webfetch, by target (ALL filtered rows,
# matching 01_headline cell 24 — webfetch is parser-robust, not no_scorer-gated).
for t in TARGET_ORDER:
    sub = [r for r in rows if r["target"] == t]
    f = sum((_num(r["toolcat_web"]) or 0) > 0 for r in sub)
    claim(f"docfetch.{t}",
          f"Fraction of '{t}' cells that fetched any external docs.",
          f"{f}/{len(sub)} = {100*f/len(sub):.0f}%",
          "count(toolcat_web > 0) / n cells, per target")
# all josh doc-fetches are llms-full.txt
nllms = sum(str(r["fetched_llms_full"]).strip() in ("1","True","true") for r in rows if r["target"] in ("josh","josh-mcp"))
nweb  = sum((_num(r["toolcat_web"]) or 0) > 0 for r in rows if r["target"] in ("josh","josh-mcp"))
claim("docfetch.llms_full",
      "Josh doc-fetches are overwhelmingly the single llms-full.txt page.",
      f"{nllms}/{nweb} Josh-arm doc-fetching cells hit llms-full.txt",
      "fetched_llms_full vs toolcat_web>0 over josh+josh-mcp")


## §Discussion — tool use / the constraint mechanism

In [ ]:
# josh-mcp has no bash: 0 arbitrary-exec / shell-compute / env-introspect.
# It compensates with a tight read->edit->run guess-and-check loop.
def mean_cat(target, cat):
    return st.mean([_num(r[f"toolcat_{cat}"]) or 0 for r in rows if r["target"]==target])
for t in TARGET_ORDER:
    claim(f"tools.{t}",
          f"Mean tool calls per '{t}' cell (edit / model-run / read / bash-categories).",
          f"edit={mean_cat(t,'edit'):.0f} model_exec={mean_cat(t,'model_exec'):.0f} "
          f"read={mean_cat(t,'read'):.0f} | bash: arb={mean_cat(t,'arbitrary_exec'):.1f} "
          f"shell={mean_cat(t,'shell_compute'):.1f} env={mean_cat(t,'env_introspect'):.1f}",
          "mean(toolcat_*) per target")
claim("tools.mcp_loop",
      "Without a shell, josh-mcp loops read->edit->run instead of introspecting the environment.",
      f"josh-mcp edits {mean_cat('josh-mcp','edit'):.0f} & model-runs {mean_cat('josh-mcp','model_exec'):.0f} per cell "
      f"(vs josh {mean_cat('josh','edit'):.0f}/{mean_cat('josh','model_exec'):.0f}); josh-mcp bash categories = 0",
      "mean toolcat edit/model_exec vs bash categories, by target")


## §Discussion — runtime within the LLM panel

In [ ]:
# Panel runtime: mesa median is fastest, but its TAIL is far worse than Josh's.
# Supports 'Josh is performant by default; no model leads you into a slow Josh model'.
for t in TARGET_ORDER:
    w = [_num(r["sim_wall_seconds"]) for r in rows
         if r["target"]==t and _bool(r["did_run"]) and _num(r["sim_wall_seconds"]) is not None]
    claim(f"runtime.{t}",
          f"Sim wall-time for target '{t}' over runnable cells (median / max).",
          f"median {median(w):.0f}s, max {max(w):.0f}s (n={len(w)})",
          "median/max(sim_wall_seconds) over did_run cells, per target")

wj = [_num(r["sim_wall_seconds"]) for r in rows if r["target"]=="josh" and _bool(r["did_run"]) and _num(r["sim_wall_seconds"]) is not None]
wm = [_num(r["sim_wall_seconds"]) for r in rows if r["target"]=="mesa" and _bool(r["did_run"]) and _num(r["sim_wall_seconds"]) is not None]
claim("runtime.no_slow_josh",
      "No LLM was led into a pathologically slow Josh model; Mesa's tail is much worse (see 02_runtime_outliers).",
      f"Josh worst-case {max(wj):.0f}s vs Mesa worst-case {max(wm):.0f}s ({max(wm)/max(wj):.1f}x)",
      "max(sim_wall_seconds) josh vs mesa over did_run cells")


## Out of scope here (separate experiments)

These manuscript numbers are **not** computed from this panel and must be cited
from their own source:

- **Wall-clock benchmark (23% non-threaded / 45% threaded faster):** a separate
  hand-written Josh-vs-Mesa runtime experiment on an AWS `m7i.2xlarge`
  (8 vCPU), 100 replicates x 100 steps at 1 km, 10 repeats. Not LLM-authored;
  not in `aggregated.csv`.
- The per-run slow-Mesa diagnosis (which pathologies cause the tail) is narrated
  in `02_runtime_outliers.ipynb`.


In [ ]:
# --- ledger dump: every claim in one table, for copy-checking against the .tex ---
print(f"{'TAG':28} VALUE")
print("-"*100)
for c in CLAIMS:
    print(f"{c['tag']:28} {c['value']}")
print(f"\nTotal claims generated: {len(CLAIMS)}")
